In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# CONFIG
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_FOLDS = 5
SEED = 2025

def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

#PATHS
LABELS_PATH = './data/labels.csv'
TRAIN_DIR = './data/train'
TEST_DIR = './data/test'

# DATASET
class HybridDataset(Dataset):
    def __init__(self, df, root_dir, augment=False):
        self.data = []
        self.labels = []
        self.fnames = []
        self.augment = augment
        
        # Mapping classes 1,2,3,6,7 to 0,1,2,3,4
        self.label_map = {1:0, 2:1, 3:2, 6:3, 7:4}
        
        print(f"Loading {len(df)} images...")
        for i in tqdm(range(len(df))):
            try:
                fname = df.iloc[i]['filename']
                path = os.path.join(root_dir, fname)
                if not os.path.exists(path):
                    path = path + '.npy'
                
                img = np.load(path).astype(np.float32)
                
                # Normalize
                img = (img - img.mean()) / (img.std() + 1e-8)
                
                # Correct shape if needed
                if img.shape[-1] == 48: 
                    img = np.transpose(img, (2, 0, 1))
                elif img.shape[0] != 48:
                    img = np.transpose(img, (2, 0, 1))

                label_raw = df.iloc[i]['label']
                if label_raw == -1:
                    label = -1
                else:
                    label = self.label_map[label_raw]

                self.data.append(img)
                self.labels.append(label)
                self.fnames.append(fname)
            except:
                continue
                
        self.data = np.array(self.data)
        self.labels = np.array(self.labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]
        label = self.labels[idx]

        if self.augment:
            k = np.random.randint(0, 4)
            img = np.rot90(img, k, axes=(1, 2)).copy()
            if np.random.random() > 0.5:
                img = np.flip(img, axis=2).copy()

        return torch.tensor(img).unsqueeze(0), torch.tensor(label).long()

# MODEL
class HybridSN(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, (7, 3, 3)), nn.BatchNorm3d(8), nn.ReLU(),
            nn.Conv3d(8, 16, (5, 3, 3)), nn.BatchNorm3d(16), nn.ReLU(),
            nn.Conv3d(16, 32, (3, 3, 3)), nn.BatchNorm3d(32), nn.ReLU()
        )
        self.conv2d = nn.Sequential(
            nn.Conv2d(32 * 36, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv3d(x)
        b, c, d, h, w = x.size()
        x = x.view(b, c * d, h, w)
        x = self.conv2d(x)
        return self.fc(x)

# TRAINING LOOP
def train_kfolds():
    df_full = pd.read_csv(LABELS_PATH)
    # Filter specific classes
    df = df_full[df_full['label'].isin([1, 2, 3, 6, 7])].reset_index(drop=True)
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    
    # Preload all data once
    full_ds = HybridDataset(df, TRAIN_DIR, augment=False)
    X_all = full_ds.data
    y_all = full_ds.labels
    
    test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith('.npy')])
    test_df = pd.DataFrame({'filename': test_files, 'label': -1})
    test_ds = HybridDataset(test_df, TEST_DIR, augment=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    oof_preds = []
    test_probs_sum = np.zeros((len(test_ds), 5))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
        print(f"FOLD {fold+1}/{N_FOLDS}")
        
        # Manual Subset
        X_train, y_train = X_all[train_idx], y_all[train_idx]
        X_val, y_val = X_all[val_idx], y_all[val_idx]
        
        # Simple Datasets wrapping the arrays
        class ArrayDataset(Dataset):
            def __init__(self, x, y, augment=False):
                self.x, self.y, self.augment = x, y, augment
            def __len__(self): return len(self.x)
            def __getitem__(self, i):
                img = self.x[i]
                if self.augment:
                    k = np.random.randint(0, 4)
                    img = np.rot90(img, k, axes=(1, 2)).copy()
                    if np.random.random() > 0.5: img = np.flip(img, axis=2).copy()
                return torch.tensor(img).unsqueeze(0), torch.tensor(self.y[i]).long()

        train_loader = DataLoader(ArrayDataset(X_train, y_train, augment=True), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ArrayDataset(X_val, y_val, augment=False), batch_size=BATCH_SIZE, shuffle=False)
        
        model = HybridSN(num_classes=5).to(DEVICE)
        optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
        criterion = nn.CrossEntropyLoss()
        
        best_acc = 0
        
        for ep in range(EPOCHS):
            model.train()
            for x, y in train_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                out = model(x)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()
            scheduler.step()
            
            # Val
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(DEVICE), y.to(DEVICE)
                    p = model(x).argmax(1)
                    correct += (p == y).sum().item()
                    total += y.size(0)
            acc = correct / total
            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), f'hybrid_fold_{fold}.pth')
        
        # Inference on Test with Best Model of Fold
        model.load_state_dict(torch.load(f'hybrid_fold_{fold}.pth'))
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for x, _ in test_loader:
                x = x.to(DEVICE)
                # TTA
                p1 = torch.softmax(model(x), 1)
                p2 = torch.softmax(model(torch.flip(x, [4])), 1) # Flip width
                p3 = torch.softmax(model(torch.rot90(x, 1, [3,4])), 1) # Rot90
                avg = (p1 + p2 + p3) / 3.0
                fold_probs.append(avg.cpu().numpy())
        
        test_probs_sum += np.concatenate(fold_probs, axis=0)

    # Final Averaging
    final_probs = test_probs_sum / N_FOLDS
    
    # CALIBRATION
    print("Calibrating...")
    inv_map = {0:1, 1:2, 2:3, 3:6, 4:7}
    target_counts = {1: 850, 2: 3100, 3: 750, 6: 13500, 7: 2063}
    priors = np.ones(5)
    
    for _ in range(50):
        preds = np.argmax(final_probs * priors, axis=1)
        mapped = [inv_map[k] for k in preds]
        counts = pd.Series(mapped).value_counts().sort_index()
        for idx, lbl in enumerate([1, 2, 3, 6, 7]):
            current_count = counts.get(lbl, 1)
            target = target_counts[lbl]
            priors[idx] *= (target / current_count) ** 0.5
            
    final_preds = np.argmax(final_probs * priors, axis=1)
    final_labels = [inv_map[k] for k in final_preds]
    
    submission = pd.DataFrame({
        'filename': test_ds.fnames,
        'label': final_labels
    })
    
    submission.to_csv('submission.csv', index=False)
    print("Saved submission.csv")
    print(submission['label'].value_counts())

if __name__ == "__main__":
    train_kfolds()